In [1]:
from crosscoder import CrossCoder
import plotly.express as px
import torch
from constants import HF_CROSSCODER_REPO

torch.set_grad_enabled(False); # for memory reduction

In [2]:
cross_coder = CrossCoder.load_from_hf()

In [3]:
norms = cross_coder.W_dec.norm(dim=-1)
print(norms.shape)
relative_norms = norms[:, 1] / norms.sum(dim=-1)
print(relative_norms.shape)

torch.Size([14336, 2])
torch.Size([14336])


In [ ]:
# Extract IT-specific, Base-specific, and Shared latent indices for SAE Vis analysis

it_specific_latent_ids = torch.where(relative_norms >= 0.9)[0]

base_specific_latent_ids = torch.where(relative_norms <= 0.1)[0]
 
shared_mask = (relative_norms >= 0.5) & (relative_norms <= 0.55)
shared_latent_ids_all = torch.where(shared_mask)[0]

# Sample 100 random shared latents from the full shared set
num_samples = 100
if len(shared_latent_ids_all) > num_samples:
    sampled_indices = torch.randperm(len(shared_latent_ids_all))[:num_samples]
    shared_latent_ids = shared_latent_ids_all[sampled_indices]

it_specific_latents = it_specific_latent_ids.tolist()
base_specific_latents = base_specific_latent_ids.tolist()
shared_latents = shared_latent_ids.tolist()

print(f"Found {len(it_specific_latents)} IT-specific latents")
print("IT-specific latent indices:", it_specific_latents)

print(f"Found {len(base_specific_latents)} Base-specific latents")
print("Base-specific latent indices:", base_specific_latents)

print(f"Sampled {len(shared_latents)} shared latents")
print("Shared latent indices (sampled from relative norm 0.5–0.55):", shared_latents)

Found 30 IT-specific latents
IT-specific latent indices: [975, 1785, 1873, 2339, 2365, 2955, 3266, 3868, 4232, 4249, 4855, 5684, 6743, 6776, 6864, 7054, 7687, 8007, 8032, 8064, 9086, 9394, 9459, 10045, 10362, 10997, 11803, 12959, 13187, 13438]
Found 8 Base-specific latents
Base-specific latent indices: [1839, 2023, 3139, 6777, 9835, 12503, 12672, 13268]
Sampled 100 shared latents
Shared latent indices (sampled from relative norm 0.5–0.55): [12012, 5067, 6177, 6198, 13661, 4129, 10388, 2026, 2836, 6443, 11996, 13864, 978, 6553, 8975, 13124, 6945, 5, 13922, 10793, 7281, 156, 6985, 9240, 13711, 9617, 6698, 10891, 12411, 10636, 5263, 2853, 7453, 10197, 12404, 6888, 13098, 3264, 2041, 6341, 3823, 6411, 503, 3725, 9715, 7614, 2378, 9699, 1602, 1265, 6538, 8778, 3917, 13328, 4067, 11612, 4383, 2757, 6925, 421, 6031, 3357, 5649, 2477, 9085, 7174, 2767, 13650, 4805, 2755, 109, 12693, 11566, 8151, 7119, 2905, 8827, 7224, 10522, 5083, 574, 10307, 3758, 4255, 5968, 4131, 5614, 12836, 49, 2540, 111

In [40]:
# Imports and constants for histograms
import numpy as np
import plotly.graph_objects as go
data = relative_norms.detach().cpu().numpy()

categories = {
    'Citi': {
        'mask': (data > 0.1) & (data < 0.4) | (data > 0.6) & (data <= 0.9),
        'color': 'gray'
    },
    'IT modeļa': {
        'mask': data >= 0.9,
        'color': 'blue'
    },
    'Bāzes modeļa': {
        'mask': data <= 0.1,
        'color': 'green'
    },
    'Kopīgie': {
        'mask': (data >= 0.4) & (data <= 0.6),
        'color': 'orange'
    },
}


In [ ]:
# Decoder norm histogram
bins = np.linspace(0, 1, 61)
bin_centers = (bins[:-1] + bins[1:]) / 2

traces = []
for name, info in categories.items():
    hist, _ = np.histogram(data[info['mask']], bins=bins)
    traces.append(go.Bar(
        x=bin_centers,
        y=hist,
        name=name,
        marker_color=info['color']
    ))

fig = go.Figure(data=traces)

fig.update_layout(
    barmode='stack',
    title=HF_CROSSCODER_REPO,
    xaxis_title="Relatīvās dekoderu latentu normu starpības",
    yaxis_title="Latentu skaits",
    showlegend=True,
    width=900,
    height=500  
)

fig.update_yaxes(
    type="log",
    tickvals=[10**i for i in range(0, 6)],
    ticktext=[f"10<sup>{i}</sup>" for i in range(0, 6)]
)

fig.update_xaxes(
    tickvals=[0.0, 0.1, 0.4, 0.6, 0.9, 1.0],
    ticktext=['0.0', '0.1', '0.4', '0.6', '0.9', '1.0']
)
fig.update_layout(
    shapes=[
        dict(
            type="line",
            x0=0.1,
            y0=0,
            x1=0.1,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="green", width=2),
        ),
        dict(
            type="line",
            x0=0.9,
            y0=0,
            x1=0.9,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="blue", width=2),
        ),
        dict(
            type="line",
            x0=0.4,
            y0=0,
            x1=0.4,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="orange", width=2),
        ),
        dict(
            type="line",
            x0=0.6,
            y0=0,
            x1=0.6,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="orange", width=2),
        ),
    ],
)

fig.show()

In [43]:
# Compute cosine similarities
cosine_sims = (
    (cross_coder.W_dec[:, 0, :] * cross_coder.W_dec[:, 1, :]).sum(dim=-1)
    / (cross_coder.W_dec[:, 0, :].norm(dim=-1) * cross_coder.W_dec[:, 1, :].norm(dim=-1))
)
cosine_data = cosine_sims.detach().cpu().numpy()

bins = np.linspace(-1, 1, 30)
bin_centers = (bins[:-1] + bins[1:]) / 2

# Normalizing count for latents for visability
traces = []
for name, info in categories.items():
    selected_data = cosine_data[info['mask']]
    hist, _ = np.histogram(selected_data, bins=bins)
    if hist.sum() > 0:
        hist = hist / hist.sum()  # Normalize to sum to 1
    traces.append(go.Bar(
        x=bin_centers,
        y=hist,
        name=name,
        marker_color=info['color']
    ))

fig = go.Figure(data=traces)

fig.update_layout(
    barmode='group',
    title=f"{HF_CROSSCODER_REPO} — Dekoderu kosinusu līdzības sadalījums",
    xaxis_title="Kosinusa līdzība starp dekoderu vektoriem",
    yaxis_title="Latentu skaits normalizēts",
    showlegend=True,
    width=900,
    height=500  
)

fig.update_yaxes(range=[0, 1])

fig.update_xaxes(
    tickvals=[-1.0, -0.5, 0.0, 0.5, 1.0],
    ticktext=['-1.0', '-0.5', '0.0', '0.5', '1.0']
)

fig.show()
